In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime

print("=" * 100)
print("🚀 INICIANDO PIPELINE DE CARGA")
print(f"⏰ Data/Hora: {datetime.now()}")
print("=" * 100)

# ------------------------------------------------------------------------------
# Parâmetros recebidos
# ------------------------------------------------------------------------------

dbutils.widgets.text("catalog_name", "catalog_carga_dev")

catalog = dbutils.widgets.get("catalog_name").strip()

if not catalog:
    raise Exception(
        "❌ O parâmetro catalog_name não foi recebido pelo Job."
    )

print(f"📦 Catálogo recebido: {catalog}")

# ------------------------------------------------------------------------------
# Catálogos disponíveis
# ------------------------------------------------------------------------------

print("\n📚 Catálogos disponíveis no Workspace:")

catalogs_df = spark.sql("SHOW CATALOGS")
catalogs_df.show(truncate=False)

catalogs = [row.catalog for row in catalogs_df.collect()]

if catalog not in catalogs:
    raise Exception(
        f"❌ Catálogo '{catalog}' não encontrado no Workspace."
    )

print(f"✅ Catálogo '{catalog}' encontrado.")

# ------------------------------------------------------------------------------
# Seleciona catálogo
# ------------------------------------------------------------------------------

spark.sql(f"USE CATALOG `{catalog}`")

active_catalog = spark.sql(
    "SELECT current_catalog()"
).collect()[0][0]

print(f"✅ Catálogo ativo: {active_catalog}")

# ------------------------------------------------------------------------------
# Schemas
# ------------------------------------------------------------------------------

print("\n📂 Schemas disponíveis:")

schemas_df = spark.sql("SHOW SCHEMAS")
schemas_df.show(truncate=False)

schemas = [row[0] for row in schemas_df.collect()]

required_schemas = ["student", "course"]

for schema in required_schemas:
    if schema not in schemas:
        raise Exception(
            f"❌ Schema obrigatório não encontrado: {schema}"
        )

print("✅ Schemas obrigatórios encontrados")

# ------------------------------------------------------------------------------
# Diagnóstico adicional
# ------------------------------------------------------------------------------

print("\n🔍 Diagnóstico do ambiente")

spark.sql("SELECT current_catalog()").show(truncate=False)
spark.sql("SELECT current_schema()").show(truncate=False)

print("✅ Diagnóstico concluído")

In [ ]:
student_data = [
    ("1", "Ana Silva", "C01"),
    ("2", "Bruno Costa", "C02"),
    ("3", "Carlos Souza", "C01"),
    ("4", "Daniela Lima", "C03")
]

student_schema = StructType([
    StructField("student_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("course_id", StringType(), True)
])

df_students = spark.createDataFrame(
    student_data,
    schema=student_schema
)

target_student_table = (
    f"{catalog}.student.tbl_student_info"
)

print("=" * 100)
print("👨‍🎓 CARGA DE ESTUDANTES")
print("=" * 100)
print(f"📍 Destino: {target_student_table}")
print(f"📊 Registros: {df_students.count()}")

df_students.write \
    .mode("overwrite") \
    .saveAsTable(target_student_table)

print("✅ Tabela gravada com sucesso")

print("\n📋 Prévia dos dados")

spark.table(target_student_table).show(
    truncate=False
)

print("✅ Validação concluída")

In [ ]:
course_data = [
    ("C01", "Engenharia de Dados"),
    ("C02", "Ciência de Dados"),
    ("C03", "Arquitetura de Nuvem")
]

course_schema = StructType([
    StructField("course_id", StringType(), True),
    StructField("course_name", StringType(), True)
])

df_courses = spark.createDataFrame(
    course_data,
    schema=course_schema
)

target_course_table = (
    f"{catalog}.course.tbl_course_info"
)

print("=" * 100)
print("📚 CARGA DE CURSOS")
print("=" * 100)
print(f"📍 Destino: {target_course_table}")
print(f"📊 Registros: {df_courses.count()}")

df_courses.write \
    .mode("overwrite") \
    .saveAsTable(target_course_table)

print("✅ Tabela gravada com sucesso")

print("\n📋 Prévia dos dados")

spark.table(target_course_table).show(
    truncate=False
)

print("✅ Validação concluída")

In [ ]:
print("=" * 100)
print("🎯 AUDITORIA FINAL")
print("=" * 100)

print(f"📦 Catálogo utilizado: {catalog}")

print("\n📂 Schemas:")

spark.sql("SHOW SCHEMAS").show(
    truncate=False
)

print("\n📋 Tabelas Student")

spark.sql(
    f"SHOW TABLES IN {catalog}.student"
).show(truncate=False)

print("\n📋 Tabelas Course")

spark.sql(
    f"SHOW TABLES IN {catalog}.course"
).show(truncate=False)

students = spark.table(
    f"{catalog}.student.tbl_student_info"
).count()

courses = spark.table(
    f"{catalog}.course.tbl_course_info"
).count()

print("\n📊 Resumo")

print(f"Alunos carregados : {students}")
print(f"Cursos carregados : {courses}")
print("=" * 100)